# 02 - Grilla Cali y Covariables Sentinel-2

Este notebook prepara la grilla espacial y las covariables Sentinel-2 que se usarán en la Situación 3. El objetivo es dejar listas las tablas necesarias para conectar estaciones de monitoreo, celdas espaciales y fechas satelitales.

## Objetivos

- Validar el panel Sentinel-2 con métricas SCL de calidad.
- Construir la grilla base de Cali a partir de `grid_id`, `lat`, `lon` y `bbox`.
- Crear banderas de calidad para nubes y píxeles no aptos.
- Asociar cada estación DAGMA/SISAIRE a la celda Sentinel-2 más cercana.
- Crear un puente temporal entre fechas diarias DAGMA y fechas disponibles de Sentinel-2.
- Guardar outputs reproducibles para los siguientes notebooks de Kriging y modelado.

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

## 1. Configuración

Se definen rutas y parámetros de calidad. Los umbrales siguen el manifest del pipeline Sentinel-2/SCL.

In [2]:
ROOT = Path('/workspace/geovision-cali-hf')

S2_PATH = ROOT / 'data/sentinel2_pipeline_reproducibilidad/outputs/sentinel2_features_cali_2020_2024_with_scl.parquet'
S2_MANIFEST_PATH = ROOT / 'data/sentinel2_pipeline_reproducibilidad/outputs/manifest_sentinel2_with_scl.json'
DAGMA_DAILY_LONG_PATH = ROOT / 'outputs/situacion3/01_panel_dagma/dagma_sisaire_daily_long.parquet'
STATION_SUMMARY_PATH = ROOT / 'outputs/situacion3/01_panel_dagma/station_summary.csv'

OUT_DIR = ROOT / 'outputs/situacion3/02_grilla_s2_features'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BBOX = [-76.60, 3.30, -76.40, 3.55]
GRID_RESOLUTION_DEGREES = 0.005
CLOUD_THRESHOLD_PCT = 30
BAD_PIXEL_THRESHOLD_PCT = 30
MAX_TEMPORAL_DISTANCE_DAYS = 30

S2_PATH, OUT_DIR

(PosixPath('/workspace/geovision-cali-hf/data/sentinel2_pipeline_reproducibilidad/outputs/sentinel2_features_cali_2020_2024_with_scl.parquet'),
 PosixPath('/workspace/geovision-cali-hf/outputs/situacion3/02_grilla_s2_features'))

## 2. Carga De Datos

Se cargan las fuentes necesarias: Sentinel-2 con SCL, el panel diario DAGMA/SISAIRE y el resumen de estaciones.

In [3]:
s2 = pd.read_parquet(S2_PATH)
dagma_daily = pd.read_parquet(DAGMA_DAILY_LONG_PATH)
station_summary_raw = pd.read_csv(STATION_SUMMARY_PATH)
s2_manifest = json.loads(S2_MANIFEST_PATH.read_text(encoding='utf-8'))

s2['date'] = pd.to_datetime(s2['date'])
dagma_daily['date'] = pd.to_datetime(dagma_daily['date'])

print('Sentinel-2:', s2.shape)
print('DAGMA diario:', dagma_daily.shape)
print('Estaciones raw:', station_summary_raw.shape)
display(s2.head())

Sentinel-2: (226352, 77)
DAGMA diario: (15653, 13)
Estaciones raw: (18, 9)


,date,year,month,grid_id,lat,lon,bbox,source,s2_cloud_pct,s2_valid_pixel_pct,s2_b1_mean,s2_b1_std,s2_b2_mean,s2_b2_std,s2_b3_mean,s2_b3_std,s2_b4_mean,s2_b4_std,s2_b5_mean,s2_b5_std,s2_b6_mean,s2_b6_std,s2_b7_mean,s2_b7_std,s2_b8_mean,s2_b8_std,s2_b8a_mean,s2_b8a_std,s2_b9_mean,s2_b9_std,s2_b11_mean,s2_b11_std,s2_b12_mean,s2_b12_std,ndvi,ndvi_std,bsi,bsi_std,ndbi,ndbi_std,vegetation_fraction,urban_index,s2_scl_total_pixel_count,s2_scl_nodata_pixel_count,s2_scl_saturated_defective_pixel_count,s2_scl_dark_area_pixel_count,s2_scl_cloud_shadow_pixel_count,s2_scl_vegetation_pixel_count,s2_scl_not_vegetated_pixel_count,s2_scl_water_pixel_count,s2_scl_unclassified_pixel_count,s2_scl_cloud_medium_probability_pixel_count,s2_scl_cloud_high_probability_pixel_count,s2_scl_thin_cirrus_pixel_count,s2_scl_snow_ice_pixel_count,s2_scl_cloud_pixel_count,s2_scl_shadow_pixel_count,s2_scl_bad_pixel_count,s2_scl_clear_pixel_count,s2_scl_nodata_pct,s2_scl_saturated_defective_pct,s2_scl_dark_area_pct,s2_scl_cloud_shadow_pct,s2_scl_vegetation_pct,s2_scl_not_vegetated_pct,s2_scl_water_pct,s2_scl_unclassified_pct,s2_scl_cloud_medium_probability_pct,s2_scl_cloud_high_probability_pct,s2_scl_thin_cirrus_pct,s2_scl_snow_ice_pct,s2_scl_cloud_pct,s2_scl_shadow_pct,s2_scl_bad_pixel_pct,s2_scl_clear_pct,s2_scl_cloud_filter_30pct_pass,s2_scl_bad_pixel_filter_30pct_pass
0,2020-01-02,2020,1,g_000_000,3.3025,-76.5975,"[-76.6, 3.3, -76.595, 3.305]",Sentinel-2 L2A / Copernicus Data Space Ecosystem,44.11,100.0,0.154174,0.012816,0.142830,0.014927,0.147180,0.017874,0.127722,0.013811,0.162510,0.024256,0.267688,0.065858,0.304275,0.080071,0.298479,0.080073,0.323149,0.088271,0.322742,0.082543,0.192549,0.038054,0.139015,0.016512,0.383633,0.091706,-0.154264,0.032755,-0.207277,0.041498,0.716049,-0.180770,729,0,0,0,499,229,1,0,0,0,0,0,0,0,499,499,230,0.0,0.0,0.0,68.449928,31.412895,0.137174,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,68.449928,68.449928,31.550068,True,False
1,2020-01-02,2020,1,g_000_001,3.3025,-76.5925,"[-76.595, 3.3, -76.59, 3.305]",Sentinel-2 L2A / Copernicus Data Space Ecosystem,44.11,100.0,0.183281,0.048957,0.166441,0.050982,0.172886,0.050197,0.155396,0.047311,0.195706,0.054842,0.284099,0.079213,0.310610,0.084635,0.301338,0.086252,0.327978,0.091073,0.348227,0.096220,0.221407,0.060122,0.168217,0.042172,0.314902,0.099725,-0.108032,0.041911,-0.150671,0.052910,0.456790,-0.129352,756,0,0,0,389,172,2,0,12,179,2,0,0,181,389,570,174,0.0,0.0,0.0,51.455025,22.751324,0.264550,0.0,1.587302,23.677248,0.264550,0.0,0.0,23.941799,51.455025,75.396828,23.015873,True,False
2,2020-01-02,2020,1,g_000_002,3.3025,-76.5875,"[-76.59, 3.3, -76.585, 3.305]",Sentinel-2 L2A / Copernicus Data Space Ecosystem,44.11,100.0,0.199862,0.079558,0.165715,0.046765,0.178302,0.045916,0.159244,0.047257,0.209981,0.051983,0.329691,0.074959,0.366422,0.084229,0.350157,0.086048,0.389104,0.091601,0.409235,0.108812,0.244759,0.052514,0.180674,0.041167,0.370303,0.118381,-0.122109,0.049473,-0.172864,0.060645,0.703704,-0.147486,756,0,0,0,269,257,0,0,25,185,20,0,0,205,269,474,257,0.0,0.0,0.0,35.582012,33.994709,0.000000,0.0,3.306878,24.470900,2.645503,0.0,0.0,27.116402,35.582012,62.698414,33.994709,True,False
3,2020-01-02,2020,1,g_000_003,3.3025,-76.5825,"[-76.585, 3.3, -76.58, 3.305]",Sentinel-2 L2A / Copernicus Data Space Ecosystem,44.11,100.0,0.383904,0.161512,0.342603,0.166974,0.338708,0.159142,0.323703,0.156325,0.364310,0.161192,0.426822,0.151868,0.449649,0.148278,0.423133,0.143090,0.465234,0.148875,0.498573,0.156566,0.376301,0.144245,0.308783,0.124585,0.163716,0.082700,-0.051416,0.032548,-0.068313,0.044612,0.055556,-0.059864,756,0,0,0,42,0,0,0,0,407,307,0,0,714,42,756,0,0.0,0.0,0.0,5.555555,0.000000,0.000000,0.0,0.000000,53.835979,40.608467,0.0,0.0,94.444443,5.555555,100.000000,0.000000,False,False
4,2020-01-02,2020,1,g_000_004,3.3025,-76.5775,"[-76.58, 3.3, -76.575, 3.305]",Sentinel-2 L2A / Copernicus Data Space Ecosystem,44.11,100.0,0.349741,0.159276,0.311311,0.148652,0.315580,0.138300,0.296411,0.1

## 3. Validaciones Iniciales

Se revisa que las columnas principales existan, que no haya duplicados por fecha y celda, y que la grilla cubra el área esperada.

In [4]:
required_s2_cols = [
    'date', 'year', 'month', 'grid_id', 'lat', 'lon', 'bbox',
    's2_scl_cloud_pct', 's2_scl_bad_pixel_pct', 's2_scl_clear_pct',
    'ndvi', 'bsi', 'ndbi', 'vegetation_fraction', 'urban_index',
]
missing_s2_cols = sorted(set(required_s2_cols) - set(s2.columns))
assert not missing_s2_cols, f'Faltan columnas Sentinel-2: {missing_s2_cols}'

duplicate_keys = int(s2.duplicated(['date', 'grid_id']).sum())
assert duplicate_keys == 0, f'Hay duplicados por date + grid_id: {duplicate_keys}'

grid_count = int(s2['grid_id'].nunique())
date_count = int(s2['date'].nunique())
spatial_extent = {
    'lat_min': float(s2['lat'].min()),
    'lat_max': float(s2['lat'].max()),
    'lon_min': float(s2['lon'].min()),
    'lon_max': float(s2['lon'].max()),
}

validation_summary = {
    's2_rows': int(len(s2)),
    's2_columns': int(s2.shape[1]),
    's2_dates': date_count,
    's2_grid_cells': grid_count,
    's2_date_min': str(s2['date'].min().date()),
    's2_date_max': str(s2['date'].max().date()),
    'duplicate_date_grid_rows': duplicate_keys,
    'spatial_extent': spatial_extent,
}
validation_summary

{'s2_rows': 226352,
 's2_columns': 77,
 's2_dates': 129,
 's2_grid_cells': 2000,
 's2_date_min': '2020-01-02',
 's2_date_max': '2024-12-16',
 'duplicate_date_grid_rows': 0,
 'spatial_extent': {'lat_min': 3.3024999999999998,
  'lat_max': 3.5475,
  'lon_min': -76.5975,
  'lon_max': -76.4025}}

## 4. Grilla Base

La grilla se obtiene del propio panel Sentinel-2 para mantener consistencia con los `grid_id` ya calculados.

In [5]:
grid = (
    s2[['grid_id', 'lat', 'lon', 'bbox']]
    .drop_duplicates()
    .sort_values('grid_id')
    .reset_index(drop=True)
)

assert len(grid) == grid_count
display(grid.head())
print('Celdas en grilla:', len(grid))

,grid_id,lat,lon,bbox
0,g_000_000,3.3025,-76.5975,"[-76.6, 3.3, -76.595, 3.305]"
1,g_000_001,3.3025,-76.5925,"[-76.595, 3.3, -76.59, 3.305]"
2,g_000_002,3.3025,-76.5875,"[-76.59, 3.3, -76.585, 3.305]"
3,g_000_003,3.3025,-76.5825,"[-76.585, 3.3, -76.58, 3.305]"
4,g_000_004,3.3025,-76.5775,"[-76.58, 3.3, -76.575, 3.305]"


Celdas en grilla: 2000


## 5. Calidad Sentinel-2/SCL

Se crea una bandera simple de calidad. Una observación pasa el filtro si tiene nubosidad y píxeles no aptos por debajo de 30%.

In [6]:
s2 = s2.copy()
s2['s2_quality_pass'] = (
    (s2['s2_scl_cloud_pct'] <= CLOUD_THRESHOLD_PCT)
    & (s2['s2_scl_bad_pixel_pct'] <= BAD_PIXEL_THRESHOLD_PCT)
)

quality_summary = (
    s2.groupby('date')
    .agg(
        grid_cells=('grid_id', 'nunique'),
        quality_pass_cells=('s2_quality_pass', 'sum'),
        cloud_pct_mean=('s2_scl_cloud_pct', 'mean'),
        cloud_pct_median=('s2_scl_cloud_pct', 'median'),
        bad_pixel_pct_mean=('s2_scl_bad_pixel_pct', 'mean'),
        bad_pixel_pct_median=('s2_scl_bad_pixel_pct', 'median'),
        clear_pct_mean=('s2_scl_clear_pct', 'mean'),
        clear_pct_median=('s2_scl_clear_pct', 'median'),
    )
    .reset_index()
)
quality_summary['quality_pass_pct'] = quality_summary['quality_pass_cells'] / quality_summary['grid_cells'] * 100

display(quality_summary.describe(include='all'))
display(quality_summary.sort_values('quality_pass_pct', ascending=False).head(10))

,date,grid_cells,quality_pass_cells,cloud_pct_mean,cloud_pct_median,bad_pixel_pct_mean,bad_pixel_pct_median,clear_pct_mean,clear_pct_median,quality_pass_pct
count,129,129.000000,129.00000,129.000000,129.000000,129.000000,129.000000,129.000000,129.000000,129.000000
mean,2022-09-11 00:00:00,1754.666667,952.24031,35.060284,26.247036,42.655289,37.905853,55.756866,59.759987,50.406458
min,2020-01-02 00:00:00,2.000000,0.00000,0.000000,0.000000,0.000574,0.000000,1.562425,0.000000,0.000000
25%,2021-07-20 00:00:00,1951.000000,574.00000,18.579287,0.000000,25.829969,0.517464,41.250542,29.846939,30.749487
50%,2022-12-22 00:00:00,1982.000000,962.00000,31.394732,6.762065,42.374283,30.994898,56.147366,64.540817,49.239643
75%,2024-01-11 00:00:00,1997.000000,1388.00000,47.529423,42.213642,56.202072,65.343918,73.287384,97.448982,69.724311
max,2024-12-16 00:00:00,2000.000000,2000.00000,97.399826,100.000000,97.561646,100.000000,99.517303,100.000000,100.000000
std,NaN,587.946258,573.02083,23.446945,35.261173,23.848816,35.966312,24.376877,36.362038,26.183534


,date,grid_cells,quality_pass_cells,cloud_pct_mean,cloud_pct_median,bad_pixel_pct_mean,bad_pixel_pct_median,clear_pct_mean,clear_pct_median,quality_pass_pct
81,2023-07-20,2000,2000,0.000000,0.0,0.000574,0.0,99.517303,100.0,100.000000
113,2024-07-29,2000,1993,0.361461,0.0,0.361461,0.0,99.203522,100.0,99.650000
116,2024-08-18,2000,1980,0.559172,0.0,0.869901,0.0,98.830925,100.0,99.000000
10,2020-03-22,2000,1968,0.819156,0.0,1.179636,0.0,98.656525,100.0,98.400000
3,2020-01-17,2000,1931,2.729599,0.0,3.081004,0.0,96.533073,100.0,96.550000
57,2022-08-24,2000,1885,3.072618,0.0,4.666230,0.0,95.048721,100.0,94.250000
17,2020-09-03,2000,1868,3.879595,0.0,5.337222,0.0,94.209839,100.0,93.400000
15,2020-08-09,2000,1864,4.780165,0.0,5.991313,0.0,93.671997,100.0,93.200000
118,2024-08-28,1999,1857,4.341763,0.0,5.826759,0.0,93.748138,100.0,92.896448
121,2024-09-17,2000,1845,5.153948,0.0,6.244382,0.0,93.371132,100.0,92.250000


## 6. Covariables Sentinel-2 Por Grilla

Se guardan dos versiones: una completa con banderas de calidad y otra filtrada por calidad. La versión completa permite auditoría; la filtrada permite modelado más controlado.

In [7]:
s2_with_quality = s2.sort_values(['date', 'grid_id']).reset_index(drop=True)
s2_quality_pass = s2_with_quality[s2_with_quality['s2_quality_pass']].copy().reset_index(drop=True)

feature_cols = [
    c for c in s2.columns
    if c.startswith('s2_b') or c in ['ndvi', 'ndvi_std', 'bsi', 'bsi_std', 'ndbi', 'ndbi_std', 'vegetation_fraction', 'urban_index']
]

feature_null_summary = (
    s2_with_quality[feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'index': 'feature', 0: 'null_fraction'})
)

print('Filas completas:', len(s2_with_quality))
print('Filas que pasan calidad:', len(s2_quality_pass))
print('Features espectrales:', len(feature_cols))
display(feature_null_summary.head(20))

Filas completas: 226352
Filas que pasan calidad: 122839
Features espectrales: 32


,feature,null_fraction
0,s2_b1_mean,0.0
1,s2_b1_std,0.0
2,vegetation_fraction,0.0
3,ndbi_std,0.0
4,ndbi,0.0
5,bsi_std,0.0
6,bsi,0.0
7,ndvi_std,0.0
8,ndvi,0.0
9,s2_b12_std,0.0


## 7. Asociación Estación-Grilla

Cada estación se asocia con la celda Sentinel-2 más cercana. Esta tabla permite cruzar observaciones reales con covariables espaciales.

In [8]:
station_summary = (
    station_summary_raw.dropna(subset=['estacion', 'lat', 'lon'])
    .groupby('estacion', as_index=False)
    .agg(
        lat=('lat', 'median'),
        lon=('lon', 'median'),
        n_obs=('n_obs', 'sum'),
        n_valid=('n_valid', 'sum'),
        n_contaminantes=('n_contaminantes', 'max'),
        min_fecha=('min_fecha', 'min'),
        max_fecha=('max_fecha', 'max'),
    )
    .sort_values('estacion')
    .reset_index(drop=True)
)

grid_coords = grid[['grid_id', 'lat', 'lon']].copy()
station_grid_rows = []

for _, station in station_summary.iterrows():
    lat0 = float(station['lat'])
    lon0 = float(station['lon'])
    dx = (grid_coords['lon'] - lon0) * 111.0 * np.cos(np.deg2rad(lat0))
    dy = (grid_coords['lat'] - lat0) * 111.0
    distance_km = np.sqrt(dx * dx + dy * dy)
    idx = int(distance_km.idxmin())
    nearest = grid_coords.loc[idx]
    station_grid_rows.append({
        'estacion': station['estacion'],
        'station_lat': lat0,
        'station_lon': lon0,
        'nearest_grid_id': nearest['grid_id'],
        'grid_lat': float(nearest['lat']),
        'grid_lon': float(nearest['lon']),
        'distance_km': float(distance_km.loc[idx]),
        'n_obs': int(station['n_obs']),
        'n_valid': int(station['n_valid']),
        'n_contaminantes': int(station['n_contaminantes']),
    })

station_to_grid = pd.DataFrame(station_grid_rows).sort_values('estacion').reset_index(drop=True)
display(station_to_grid)
print('Distancia máxima estación-grilla (km):', station_to_grid['distance_km'].max())

,estacion,station_lat,station_lon,nearest_grid_id,grid_lat,grid_lon,distance_km,n_obs,n_valid,n_contaminantes
0,Base Aerea,3.457128,-76.502303,g_031_019,3.4575,-76.5025,0.046706,101117,52160,2
1,Canaveralejo,3.416366,-76.549613,g_023_010,3.4175,-76.5475,0.265818,68248,32923,3
2,Compartir,3.428260,-76.466584,g_025_026,3.4275,-76.4675,0.131976,73424,35056,3
3,Era Obrero,3.457317,-76.506539,g_031_018,3.4575,-76.5075,0.108397,72768,26535,3
4,Ermita,3.455514,-76.530978,g_031_013,3.4575,-76.5325,0.277550,70904,36920,3
5,Flora,3.488218,-76.518058,g_037_016,3.4875,-76.5175,0.100866,131544,48718,3
6,Pance,3.304517,-76.531252,g_000_013,3.3025,-76.5325,0.263157,74016,36340,3
7,Transitoria Navarro,3.417183,-76.494960,g_023_021,3.4175,-76.4925,0.274836,109525,21054,3
8,Univalle,3.377911,-76.533811,g_015_013,3.3775,-76.5325,0.152263,101600,60225,2


Distancia máxima estación-grilla (km): 0.27755026613761125


## 8. Puente Temporal DAGMA-Sentinel-2

DAGMA/SISAIRE tiene observaciones diarias, mientras que Sentinel-2 solo está disponible en fechas de adquisición. Se crea una tabla que asigna a cada fecha diaria la fecha Sentinel-2 más cercana dentro de una ventana de 30 días.

In [9]:
dagma_dates = pd.Series(pd.to_datetime(sorted(dagma_daily['date'].dropna().unique())), name='dagma_date')
s2_dates = pd.Series(pd.to_datetime(sorted(s2['date'].dropna().unique())), name='s2_date')

s2_date_values = s2_dates.to_numpy(dtype='datetime64[ns]')
bridge_rows = []

for dagma_date in dagma_dates:
    diffs = np.abs((s2_date_values - np.datetime64(dagma_date)).astype('timedelta64[D]').astype(int))
    nearest_idx = int(diffs.argmin())
    nearest_s2_date = pd.Timestamp(s2_date_values[nearest_idx])
    distance_days = int(diffs[nearest_idx])
    bridge_rows.append({
        'dagma_date': dagma_date,
        'nearest_s2_date': nearest_s2_date,
        'temporal_distance_days': distance_days,
        'within_30_days': distance_days <= MAX_TEMPORAL_DISTANCE_DAYS,
    })

dagma_to_s2_bridge = pd.DataFrame(bridge_rows)

display(dagma_to_s2_bridge['temporal_distance_days'].describe())
display(dagma_to_s2_bridge.sort_values('temporal_distance_days', ascending=False).head(10))
print('Fechas DAGMA:', len(dagma_to_s2_bridge))
print('Fechas dentro de 30 días:', int(dagma_to_s2_bridge['within_30_days'].sum()))

count    1827.000000
mean        6.417077
std         7.012358
min         0.000000
25%         2.000000
50%         4.000000
75%         8.000000
max        40.000000
Name: temporal_distance_days, dtype: float64

,dagma_date,nearest_s2_date,temporal_distance_days,within_30_days
1046,2022-11-12,2022-10-03,40,False
1045,2022-11-11,2022-10-03,39,False
1047,2022-11-13,2022-12-22,39,False
1048,2022-11-14,2022-12-22,38,False
1044,2022-11-10,2022-10-03,38,False
1043,2022-11-09,2022-10-03,37,False
1049,2022-11-15,2022-12-22,37,False
1050,2022-11-16,2022-12-22,36,False
1042,2022-11-08,2022-10-03,36,False
451,2021-03-27,2021-02-20,35,False


Fechas DAGMA: 1827
Fechas dentro de 30 días: 1790


## 9. Guardar Outputs

Se guardan las tablas preparadas y un manifest con rutas, tamaños y hashes para trazabilidad.

In [10]:
paths = {
    'grid': OUT_DIR / 'grid_cali_005deg.parquet',
    's2_with_quality': OUT_DIR / 's2_grid_features_with_quality.parquet',
    's2_quality_pass': OUT_DIR / 's2_grid_features_quality_pass.parquet',
    's2_date_quality_summary': OUT_DIR / 's2_date_quality_summary.csv',
    'station_to_grid': OUT_DIR / 'station_to_s2_grid_mapping.csv',
    'dagma_to_s2_bridge': OUT_DIR / 'dagma_daily_to_s2_date_bridge.csv',
    'feature_null_summary': OUT_DIR / 's2_feature_null_summary.csv',
}

grid.to_parquet(paths['grid'], index=False)
s2_with_quality.to_parquet(paths['s2_with_quality'], index=False)
s2_quality_pass.to_parquet(paths['s2_quality_pass'], index=False)
quality_summary.to_csv(paths['s2_date_quality_summary'], index=False)
station_to_grid.to_csv(paths['station_to_grid'], index=False)
dagma_to_s2_bridge.to_csv(paths['dagma_to_s2_bridge'], index=False)
feature_null_summary.to_csv(paths['feature_null_summary'], index=False)

def md5_file(path):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    'notebook': '02_grilla_cali_covariables.ipynb',
    'inputs': {
        'sentinel2_with_scl': str(S2_PATH),
        'sentinel2_manifest': str(S2_MANIFEST_PATH),
        'dagma_daily_long': str(DAGMA_DAILY_LONG_PATH),
        'station_summary': str(STATION_SUMMARY_PATH),
    },
    'input_md5': {
        'sentinel2_with_scl': md5_file(S2_PATH),
        'sentinel2_manifest': md5_file(S2_MANIFEST_PATH),
        'dagma_daily_long': md5_file(DAGMA_DAILY_LONG_PATH),
        'station_summary': md5_file(STATION_SUMMARY_PATH),
    },
    'parameters': {
        'bbox': BBOX,
        'grid_resolution_degrees': GRID_RESOLUTION_DEGREES,
        'cloud_threshold_pct': CLOUD_THRESHOLD_PCT,
        'bad_pixel_threshold_pct': BAD_PIXEL_THRESHOLD_PCT,
        'max_temporal_distance_days': MAX_TEMPORAL_DISTANCE_DAYS,
    },
    'summary': {
        **validation_summary,
        'quality_pass_rows': int(len(s2_quality_pass)),
        'quality_pass_fraction': float(len(s2_quality_pass) / len(s2_with_quality)),
        'stations_mapped': int(len(station_to_grid)),
        'max_station_grid_distance_km': float(station_to_grid['distance_km'].max()),
        'dagma_dates': int(len(dagma_to_s2_bridge)),
        'dagma_dates_within_30_days': int(dagma_to_s2_bridge['within_30_days'].sum()),
        'max_temporal_distance_days_observed': int(dagma_to_s2_bridge['temporal_distance_days'].max()),
    },
    'outputs': {k: str(v) for k, v in paths.items()},
    'output_md5': {k: md5_file(v) for k, v in paths.items()},
}

manifest_path = OUT_DIR / 'manifest_02_grilla_s2_features.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

manifest

{'notebook': '02_grilla_cali_covariables.ipynb',
 'inputs': {'sentinel2_with_scl': '/workspace/geovision-cali-hf/data/sentinel2_pipeline_reproducibilidad/outputs/sentinel2_features_cali_2020_2024_with_scl.parquet',
  'sentinel2_manifest': '/workspace/geovision-cali-hf/data/sentinel2_pipeline_reproducibilidad/outputs/manifest_sentinel2_with_scl.json',
  'dagma_daily_long': '/workspace/geovision-cali-hf/outputs/situacion3/01_panel_dagma/dagma_sisaire_daily_long.parquet',
  'station_summary': '/workspace/geovision-cali-hf/outputs/situacion3/01_panel_dagma/station_summary.csv'},
 'input_md5': {'sentinel2_with_scl': '237084f71b24650c6ed1b2957c783749',
  'sentinel2_manifest': '8fb66ce82463ed8329ffd4b5f1b672c4',
  'dagma_daily_long': '581644c3b09f1aacc148388bf8f62e2f',
  'station_summary': 'b54f4f8a6a5531dc10e7151d914c0ddb'},
 'parameters': {'bbox': [-76.6, 3.3, -76.4, 3.55],
  'grid_resolution_degrees': 0.005,
  'cloud_threshold_pct': 30,
  'bad_pixel_threshold_pct': 30,
  'max_temporal_dist

## 10. Cierre

Este notebook deja preparada la grilla espacial, las features Sentinel-2 con control de calidad, el enlace estación-grilla y el puente temporal DAGMA-Sentinel-2. Estos archivos serán usados en los notebooks posteriores de Kriging, validación LOO-CV y modelado temporal.